# CELL 1: SETUP & LOAD VIA GOOGLE DRIVE

In [2]:
import os
import pandas as pd
import numpy as np
import pickle
from scipy.io import loadmat
from sklearn.preprocessing import StandardScaler
from google.colab import drive

print("Mounting Google Drive...")
# This will prompt you to click a link and allow Colab to read your Drive
drive.mount('/content/drive')

# If you put the file in your main Drive folder, this is the exact path:
MAT_FILE_PATH = '/content/drive/MyDrive/Oxford_Battery_Degradation_Dataset_1.mat'

print("\nLoading 262MB .mat file into RAM (This will take 1-2 minutes)...")
try:
    mat_data = loadmat(MAT_FILE_PATH)
    cell_keys = [k for k in mat_data.keys() if k.startswith('Cell')]
    print(f"✓ Successfully loaded! Found cells: {cell_keys}")
except Exception as e:
    print(f"❌ Error loading file: {e}")
    print("Please check that the file is fully uploaded to your main Google Drive folder.")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Loading 262MB .mat file into RAM (This will take 1-2 minutes)...
✓ Successfully loaded! Found cells: ['Cell1', 'Cell2', 'Cell3', 'Cell4', 'Cell5', 'Cell6', 'Cell7', 'Cell8']


# CELL 2: EXTRACTION

In [3]:
extracted_rows = []

print("Starting deep extraction of discharge cycles...")
for cell_name in cell_keys:
    print(f"\n-> Processing {cell_name}...")
    cell_struct = mat_data[cell_name]

    # Unwrap top-level MATLAB struct
    cell_data = cell_struct[0, 0] if isinstance(cell_struct, np.ndarray) and cell_struct.shape == (1, 1) else cell_struct

    cycle_keys = [k for k in cell_data.dtype.names if k.startswith('cyc')]
    print(f"   Found {len(cycle_keys)} cycles. Extracting...", end=" ")

    cycles_extracted = 0
    for cyc_name in cycle_keys:
        cycle_idx = int(cyc_name.replace('cyc', ''))
        cyc_struct = cell_data[cyc_name]

        if isinstance(cyc_struct, np.ndarray) and cyc_struct.size > 0:
            cyc_data = cyc_struct[0, 0]
            phase_keys = [k for k in cyc_data.dtype.names if 'dc' in k.lower()] # Discharge phases

            for phase_name in phase_keys:
                phase_struct = cyc_data[phase_name]
                if isinstance(phase_struct, np.ndarray) and phase_struct.size > 0:
                    phase_data = phase_struct[0, 0]

                    try:
                        time_s = phase_data['t'].flatten()
                        voltage_v = phase_data['v'].flatten()
                        temperature_c = phase_data['T'].flatten()
                        capacity_ah = phase_data['q'].flatten()

                        if 'i' in phase_data.dtype.names: current_a = phase_data['i'].flatten()
                        elif 'I' in phase_data.dtype.names: current_a = phase_data['I'].flatten()
                        else: current_a = np.zeros_like(voltage_v)

                        for i in range(len(time_s)):
                            extracted_rows.append({
                                'cell_id': cell_name, 'cycle_number': cycle_idx,
                                'time_s': time_s[i], 'voltage_V': voltage_v[i],
                                'current_A': current_a[i], 'temperature_C': temperature_c[i],
                                'capacity_Ah': capacity_ah[i]
                            })
                        cycles_extracted += 1
                    except Exception:
                        pass # Skip characterization arrays

    print(f"✓ Extracted {cycles_extracted} valid driving discharge cycles.")

df = pd.DataFrame(extracted_rows)
print(f"\n✅ Total raw rows extracted: {len(df):,}")

Starting deep extraction of discharge cycles...

-> Processing Cell1...
   Found 78 cycles. Extracting... ✓ Extracted 156 valid driving discharge cycles.

-> Processing Cell2...
   Found 73 cycles. Extracting... ✓ Extracted 146 valid driving discharge cycles.

-> Processing Cell3...
   Found 76 cycles. Extracting... ✓ Extracted 152 valid driving discharge cycles.

-> Processing Cell4...
   Found 47 cycles. Extracting... ✓ Extracted 94 valid driving discharge cycles.

-> Processing Cell5...
   Found 46 cycles. Extracting... ✓ Extracted 92 valid driving discharge cycles.

-> Processing Cell6...
   Found 46 cycles. Extracting... ✓ Extracted 92 valid driving discharge cycles.

-> Processing Cell7...
   Found 77 cycles. Extracting... ✓ Extracted 154 valid driving discharge cycles.

-> Processing Cell8...
   Found 76 cycles. Extracting... ✓ Extracted 152 valid driving discharge cycles.

✅ Total raw rows extracted: 7,669,585


# CELL 3: ENGINEERING & ZERO-LEAKAGE SCALING

In [4]:
print("Engineering RL State Variables (Preserving Spikes)...")
df['voltage_diff_V'] = df.groupby(['cell_id', 'cycle_number'])['voltage_V'].diff().fillna(0)
df['temp_diff_C'] = df.groupby(['cell_id', 'cycle_number'])['temperature_C'].diff().fillna(0)

print("Applying Zero-Leakage Standardization...")
state_features = ['voltage_V', 'current_A', 'temperature_C', 'voltage_diff_V', 'temp_diff_C', 'capacity_Ah']

# Fit scaler STRICTLY on training cells (Cells 1 to 6)
train_cells = ['Cell1', 'Cell2', 'Cell3', 'Cell4', 'Cell5', 'Cell6']
train_mask = df['cell_id'].isin(train_cells)

scaler = StandardScaler()
scaler.fit(df.loc[train_mask, state_features])

# Transform ALL cells chronologically
df_scaled = df.copy()
df_scaled[state_features] = scaler.transform(df[state_features])

print("✅ Scaling Complete!")

Engineering RL State Variables (Preserving Spikes)...
Applying Zero-Leakage Standardization...
✅ Scaling Complete!


# CELL 4: EXPORT & DOWNLOAD

In [5]:
from google.colab import files

print("Splitting into Train and Test datasets...")
# Train: Cells 1-6 | Test: Cells 7-8
train_df = df_scaled[df_scaled['cell_id'].isin(['Cell1', 'Cell2', 'Cell3', 'Cell4', 'Cell5', 'Cell6'])]
test_df = df_scaled[df_scaled['cell_id'].isin(['Cell7', 'Cell8'])]

print("Exporting to highly compressed Parquet format...")
# Parquet preserves all Pandas data types and compresses massive files
train_df.to_parquet('oxford_train_env.parquet', index=False)
test_df.to_parquet('oxford_test_env.parquet', index=False)

print("Exporting Scaler...")
import pickle
with open('oxford_rl_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Downloading files to your local machine...")
files.download('oxford_train_env.parquet')
files.download('oxford_test_env.parquet')
files.download('oxford_rl_scaler.pkl')

print("✅ Pipeline Complete! You now have sleek, professional datasets.")

Splitting into Train and Test datasets...
Exporting to highly compressed Parquet format...
Exporting Scaler...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Pipeline Complete! You now have sleek, professional datasets.
